# Ⅰ. 원본 데이터 점검

- **목적:** 전처리를 수행하기 전에 원본 데이터를 있는 그대로 살펴보고,
어떤 오류·이상값이 존재하는지 파악한 뒤 **전처리 정책**을 정의한다.

| 단계 | 내용 |
|------|------|
| 1 | 데이터 로드 |
| 2 | 기본 구조 확인 (`shape / info / columns`) |
| 3 | 결측값 확인 |
| 4 | 중복 응답 탐색 |
| 5 | 회차 오기재 가능성 확인 (응답일 vs 선택회차 불일치) |
| 6 | 직무 문자열 정규화 및 동명이인 여부 재확인 |
| 7 | 현재까지 작업한 내용 파일로 저장 |
| 8 | 전처리 정책 정의 |

## 1. 데이터 로드

In [2]:
import pandas as pd

In [3]:
RAW_DATA_PATH = "../data/raw_data.xlsx"

df_pre  = pd.read_excel(RAW_DATA_PATH, sheet_name="사전_설문_RAW", header=1)
df_post = pd.read_excel(RAW_DATA_PATH, sheet_name="사후_설문_RAW", header=1)

df_pre.head()

,응답시각,교육일,선택회차,참여자명,직무,엑셀사용경력,A1_데이터정리수치도출,A2_데이터시각화,B1_업무활용계획,B2_교육내용적용의향,C1_팀내공유의향,C2_데이터문화정착가능성
0,2025-03-05 01:13:19,2025-03-05,1회차,한채원,마케팅·광고·MD,1년 미만,보통이다,보통이다,그렇지 않다,보통이다,그렇지 않다,그렇다
1,2025-03-05 01:22:50,2025-03-05,1회차,홍은서,디자인,거의 없음,보통이다,전혀 그렇지 않다,그렇지 않다,전혀 그렇지 않다,그렇지 않다,보통이다
2,2025-03-05 04:42:28,2025-03-05,1회차,강지수,마케팅/광고,3년 이상,그렇다,그렇다,매우 그렇다,매우 그렇다,그렇다,보통이다
3,2025-03-05 07:43:04,2025-03-05,1회차,김소윤,마케팅·광고·MD,1~3년,보통이다,그렇다,보통이다,그렇지 않다,보통이다,보통이다
4,2025-03-05 08:05:20,2025-03-05,1회차,조도윤,기획·전략,1~3년,보통이다,그렇다,보통이다,그렇다,보통이다,그렇지 않다


## 2. 기본 구조 확인

In [4]:
# 행과 열 
print("사전 설문", df_pre.shape)
print("사후 설문", df_post.shape)

사전 설문 (98, 12)
사후 설문 (98, 16)


- 사전 설문과 사후 설문의 row 개수가 다름
    - 둘 다 존재하는 데이터만 갖고 분석해야됨.

In [6]:
# 데이터 구조
print("===== 사전 설문 =====")
df_pre.info()

print("\n ===== 사후 설문 =====")
df_post.info()

===== 사전 설문 =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   응답시각           98 non-null     object
 1   교육일            98 non-null     object
 2   선택회차           98 non-null     object
 3   참여자명           98 non-null     object
 4   직무             98 non-null     object
 5   엑셀사용경력         98 non-null     object
 6   A1_데이터정리수치도출   98 non-null     object
 7   A2_데이터시각화      98 non-null     object
 8   B1_업무활용계획      98 non-null     object
 9   B2_교육내용적용의향    98 non-null     object
 10  C1_팀내공유의향      98 non-null     object
 11  C2_데이터문화정착가능성  98 non-null     object
dtypes: object(12)
memory usage: 9.3+ KB

 ===== 사후 설문 =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   응답시각           98 non-

- 컬럼 타입이 object이므로 리커트 척도(숫자형)로 바꿔줘야됨.

## 3. 결측값 확인

In [ ]:
df_pre.isnull().sum() # 컬럼별 결측치 개수 - 사전 설문

응답시각             0
교육일              0
선택회차             0
참여자명             0
직무               0
엑셀사용경력           0
A1_데이터정리수치도출     0
A2_데이터시각화        0
B1_업무활용계획        0
B2_교육내용적용의향      0
C1_팀내공유의향        0
C2_데이터문화정착가능성    0
dtype: int64

In [ ]:
df_post.isnull().sum() # 컬럼별 결측치 개수 - 사후 설문

응답시각             0
교육일              0
선택회차             0
참여자명             0
직무               0
엑셀사용경력           0
A1_데이터정리수치도출     0
A2_데이터시각화        0
B1_업무활용계획        0
B2_교육내용적용의향      0
C1_팀내공유의향        0
C2_데이터문화정착가능성    0
D1_강사            0
D2_내용            0
D3_운영환경          0
D4_전반만족도         0
dtype: int64

- 결측값 확인 -> 없음

## 4. 중복 응답 탐색

### 사전 설문 (df_pre)

In [8]:
df_pre.duplicated().sum()

np.int64(0)

In [10]:
df_pre.groupby(['선택회차', '참여자명']).size().sort_values(ascending=False)

선택회차  참여자명
4회차   장시우     2
5회차   한지원     2
3회차   홍민재     2
      박지영     2
4회차   박민준     2
             ..
2회차   안민재     1
      신수아     1
      송지수     1
      송건우     1
5회차   황민재     1
Length: 91, dtype: int64

- 같은 회차, 같은 이름 응답 건수가 존재함 -> 중복 응답이 있다는 뜻

In [22]:
df_pre.groupby(['선택회차', '참여자명']).size().reset_index(name='count').query("count > 1").shape[0]

7

- 중복 응답의 참여자명 7건 있음.

In [15]:
print(len(df_pre[df_pre.duplicated(subset=['선택회차', '참여자명'], keep=False)]))

df_pre[df_pre.duplicated(subset=['선택회차', '참여자명'], keep=False)]

14


,응답시각,교육일,선택회차,참여자명,직무,엑셀사용경력,A1_데이터정리수치도출,A2_데이터시각화,B1_업무활용계획,B2_교육내용적용의향,C1_팀내공유의향,C2_데이터문화정착가능성
18,2025-03-12 00:48:44,2025-03-12,2회차,황예준,디자인,3년 이상,보통이다,그렇지 않다,매우 그렇다,매우 그렇다,보통이다,그렇지 않다
19,2025-03-12 01:04:47,2025-03-12,2회차,이준서,기획·전략,1년 미만,보통이다,전혀 그렇지 않다,그렇지 않다,전혀 그렇지 않다,전혀 그렇지 않다,그렇다
25,2025-03-12 06:26:04,2025-03-12,2회차,황예준,마케팅·광고·MD,거의 없음,전혀 그렇지 않다,그렇지 않다,그렇지 않다,그렇다,그렇지 않다,보통이다
26,2025-03-12 09:47:56,2025-03-12,2회차,이준서,기획·전략,1년 미만,보통이다,전혀 그렇지 않다,그렇지 않다,전혀 그렇지 않다,전혀 그렇지 않다,그렇다
41,2025-03-19 01:47:06,2025-03-19,3회차,홍민재,마케팅 광고,거의 없음,그렇지 않다,그렇지 않다,그렇지 않다,보통이다,보통이다,그렇다
45,2025-03-19 04:48:01,2025-03-19,3회차,박지영,영업·영업지원,3년 이상,보통이다,보통이다,매우 그렇다,매우 그렇다,보통이다,매우 그렇다
51,2025-03-19 15:32:03,2025-03-19,3회차,홍민재,마케팅,거의 없음,보통이다,보통이다,그렇지 않다,전혀 그렇지 않다,전혀 그렇지 않다,그렇지 않다
54,2025-03-19 21:14:55,2025-03-19,3회차,박지영,디자인,1년 미만,보통이다,보통이다,보통이다,보통이다,보통이다,보통이다
58,2025-03-26 01:10:46,2025-03-26,4회차,박민준,마케팅·광고·MD,1~3년,그렇지 않다,보통이다,그렇다,전혀 그렇지 않다,그렇지 않다,그렇다
61,2025-03-26 02:40:34,2025-03-26,4회차,박민준,마케팅·광고·MD,1~3년,그렇지 않다,보통이다,그렇다,전혀 그렇지 않다,그렇지 않다,그렇다


In [56]:
dup_pre = df_pre[df_pre.duplicated(subset=["선택회차", "참여자명"], keep=False)]

job_check = (
    dup_pre.groupby(["선택회차", "참여자명"])["직무"]
       .nunique()
       .reset_index(name="job_unique")
)

job_check["is_different_person"] = job_check["job_unique"] > 1

job_check["is_different_person"] 

0    False
1     True
2     True
3     True
4    False
5     True
6    False
Name: is_different_person, dtype: bool

- 동명이인 판별 기준
    - 직무 완전 다름(예: 개발 vs 인사)이면 동명이인
    - 직무 유사(마케팅 vs 마케팅 광고)는 같은 사람으로 간주

In [57]:
dup_pre = dup_pre.merge(
    job_check[["선택회차", "참여자명", "is_different_person"]],
    on=["선택회차", "참여자명"],
    how="left"
)
dup_pre

,응답시각,교육일,선택회차,참여자명,직무,엑셀사용경력,A1_데이터정리수치도출,A2_데이터시각화,B1_업무활용계획,B2_교육내용적용의향,C1_팀내공유의향,C2_데이터문화정착가능성,응답일,정상회차,is_different_person
0,2025-03-12 00:48:44,2025-03-12,2회차,황예준,디자인,3년 이상,보통이다,그렇지 않다,매우 그렇다,매우 그렇다,보통이다,그렇지 않다,2025-03-12,2회차,True
1,2025-03-12 01:04:47,2025-03-12,2회차,이준서,기획·전략,1년 미만,보통이다,전혀 그렇지 않다,그렇지 않다,전혀 그렇지 않다,전혀 그렇지 않다,그렇다,2025-03-12,2회차,False
2,2025-03-12 06:26:04,2025-03-12,2회차,황예준,마케팅·광고·MD,거의 없음,전혀 그렇지 않다,그렇지 않다,그렇지 않다,그렇다,그렇지 않다,보통이다,2025-03-12,2회차,True
3,2025-03-12 09:47:56,2025-03-12,2회차,이준서,기획·전략,1년 미만,보통이다,전혀 그렇지 않다,그렇지 않다,전혀 그렇지 않다,전혀 그렇지 않다,그렇다,2025-03-12,2회차,False
4,2025-03-19 01:47:06,2025-03-19,3회차,홍민재,마케팅 광고,거의 없음,그렇지 않다,그렇지 않다,그렇지 않다,보통이다,보통이다,그렇다,2025-03-19,3회차,True
5,2025-03-19 04:48:01,2025-03-19,3회차,박지영,영업·영업지원,3년 이상,보통이다,보통이다,매우 그렇다,매우 그렇다,보통이다,매우 그렇다,2025-03-19,3회차,True
6,2025-03-19 15:32:03,2025-03-19,3회차,홍민재,마케팅,거의 없음,보통이다,보통이다,그렇지 않다,전혀 그렇지 않다,전혀 그렇지 않다,그렇지 않다,2025-03-19,3회차,True
7,2025-03-19 21:14:55,2025-03-19,3회차,박지영,디자인,1년 미만,보통이다,보통이다,보통이다,보통이다,보통이다,보통이다,2025-03-19,3회차,True
8,2025-03-26 01:10:46,2025-03-26,4회차,박민준,마케팅·광고·MD,1~3년,그렇지 않다,보통이다,그렇다,전혀 그렇지 않다,그렇지 않다,그렇다,2025-03-26,4회차,False
9,2025-03-26 02:40:34,2025-03-26,4회차,박민준,마케팅·광고·MD,1~3년,그렇지 않다,보통이다,그렇다,전혀 그렇지 않다,그렇지 않다,그렇다,2025-03-26,4회차,False


- 중복 응답 14개 (7쌍 x 2)
    - is different person : false 인 행은 더 최신 응답만 남겨두고 제거
    - 그 중 홍민재-마케팅 광고, 홍민재-마케팅 같은 응답은 동명이인으로 판단
- 이를 보니 직무 문자열 정규화가 필요함 -> 추후 분포 탐색 이전에 진행

### 사후 설문 (df_post)

In [16]:
df_post.duplicated().sum()

np.int64(0)

In [19]:
df_post.groupby(['선택회차', '참여자명']).size().sort_values(ascending=False)

선택회차  참여자명
2회차   황예준     2
5회차   한지원     2
4회차   박민준     2
      장시우     2
3회차   박지영     2
             ..
2회차   안지수     1
      안민재     1
      신수아     1
      신다은     1
5회차   황민재     1
Length: 91, dtype: int64

- 사후 설문도 중복 응답 존재

In [24]:
df_post.groupby(['선택회차', '참여자명']).size().reset_index(name='count').query("count > 1").shape[0]

7

- 중복 응답의 참여자명 7건 있음.

In [20]:
print(len(df_post[df_post.duplicated(subset=['선택회차', '참여자명'], keep=False)]))

df_post[df_post.duplicated(subset=['선택회차', '참여자명'], keep=False)]

14


,응답시각,교육일,선택회차,참여자명,직무,엑셀사용경력,A1_데이터정리수치도출,A2_데이터시각화,B1_업무활용계획,B2_교육내용적용의향,C1_팀내공유의향,C2_데이터문화정착가능성,D1_강사,D2_내용,D3_운영환경,D4_전반만족도
21,2025-03-12 03:59:20,2025-03-12,2회차,이준서,기획·전략,1년 미만,매우 그렇다,그렇지 않다,그렇지 않다,전혀 그렇지 않다,그렇지 않다,매우 그렇다,그렇다,그렇다,그렇다,보통이다
28,2025-03-12 12:55:52,2025-03-12,2회차,황예준,마케팅·광고·MD,거의 없음,그렇지 않다,보통이다,보통이다,그렇다,보통이다,매우 그렇다,그렇다,보통이다,그렇다,그렇다
29,2025-03-12 15:16:20,2025-03-12,2회차,황예준,디자인,3년 이상,그렇다,보통이다,매우 그렇다,그렇다,그렇다,보통이다,보통이다,매우 그렇다,매우 그렇다,그렇다
30,2025-03-12 16:08:14,2025-03-12,2회차,이준서,기획·전략,1년 미만,매우 그렇다,그렇지 않다,그렇지 않다,전혀 그렇지 않다,그렇지 않다,매우 그렇다,그렇다,그렇다,그렇다,보통이다
40,2025-03-19 00:37:11,2025-03-19,3회차,홍민재,마케팅,거의 없음,그렇다,보통이다,보통이다,그렇다,그렇다,그렇다,매우 그렇다,그렇다,그렇다,그렇다
42,2025-03-19 02:16:41,2025-03-19,3회차,홍민재,광고·MD,거의 없음,그렇다,매우 그렇다,그렇다,그렇지 않다,전혀 그렇지 않다,보통이다,그렇다,그렇다,그렇다,그렇다
44,2025-03-19 07:18:28,2025-03-19,3회차,박지영,영업·영업지원,3년 이상,그렇다,매우 그렇다,매우 그렇다,매우 그렇다,그렇다,매우 그렇다,매우 그렇다,그렇다,매우 그렇다,그렇다
51,2025-03-19 12:51:46,2025-03-19,3회차,박지영,디자인,1년 미만,보통이다,그렇다,그렇다,보통이다,매우 그렇다,매우 그렇다,매우 그렇다,그렇다,매우 그렇다,그렇다
63,2025-03-26 04:00:09,2025-03-26,4회차,박민준,마케팅·광고·MD,1~3년,보통이다,그렇다,매우 그렇다,그렇지 않다,보통이다,매우 그렇다,그렇다,그렇다,그렇다,그렇다
67,2025-03-26 08:08:03,2025-03-26,4회차,박민준,마케팅·광고·MD,1~3년,보통이다,그렇다,매우 그렇다,그렇지 않다,보통이다,매우 그렇다,그렇다,그렇다,그렇다,그렇다


In [58]:
dup_post = df_post[df_post.duplicated(subset=["선택회차", "참여자명"], keep=False)]

job_check = (
    dup_post.groupby(["선택회차", "참여자명"])["직무"]
       .nunique()
       .reset_index(name="job_unique")
)

job_check["is_different_person"] = job_check["job_unique"] > 1

job_check["is_different_person"] 

0    False
1     True
2     True
3     True
4    False
5     True
6    False
Name: is_different_person, dtype: bool

In [59]:
dup_post = dup_post.merge(
    job_check[["선택회차", "참여자명", "is_different_person"]],
    on=["선택회차", "참여자명"],
    how="left"
)
dup_post

,응답시각,교육일,선택회차,참여자명,직무,엑셀사용경력,A1_데이터정리수치도출,A2_데이터시각화,B1_업무활용계획,B2_교육내용적용의향,C1_팀내공유의향,C2_데이터문화정착가능성,D1_강사,D2_내용,D3_운영환경,D4_전반만족도,응답일,정상회차,is_different_person
0,2025-03-12 03:59:20,2025-03-12,2회차,이준서,기획·전략,1년 미만,매우 그렇다,그렇지 않다,그렇지 않다,전혀 그렇지 않다,그렇지 않다,매우 그렇다,그렇다,그렇다,그렇다,보통이다,2025-03-12,2회차,False
1,2025-03-12 12:55:52,2025-03-12,2회차,황예준,마케팅·광고·MD,거의 없음,그렇지 않다,보통이다,보통이다,그렇다,보통이다,매우 그렇다,그렇다,보통이다,그렇다,그렇다,2025-03-12,2회차,True
2,2025-03-12 15:16:20,2025-03-12,2회차,황예준,디자인,3년 이상,그렇다,보통이다,매우 그렇다,그렇다,그렇다,보통이다,보통이다,매우 그렇다,매우 그렇다,그렇다,2025-03-12,2회차,True
3,2025-03-12 16:08:14,2025-03-12,2회차,이준서,기획·전략,1년 미만,매우 그렇다,그렇지 않다,그렇지 않다,전혀 그렇지 않다,그렇지 않다,매우 그렇다,그렇다,그렇다,그렇다,보통이다,2025-03-12,2회차,False
4,2025-03-19 00:37:11,2025-03-19,3회차,홍민재,마케팅,거의 없음,그렇다,보통이다,보통이다,그렇다,그렇다,그렇다,매우 그렇다,그렇다,그렇다,그렇다,2025-03-19,3회차,True
5,2025-03-19 02:16:41,2025-03-19,3회차,홍민재,광고·MD,거의 없음,그렇다,매우 그렇다,그렇다,그렇지 않다,전혀 그렇지 않다,보통이다,그렇다,그렇다,그렇다,그렇다,2025-03-19,3회차,True
6,2025-03-19 07:18:28,2025-03-19,3회차,박지영,영업·영업지원,3년 이상,그렇다,매우 그렇다,매우 그렇다,매우 그렇다,그렇다,매우 그렇다,매우 그렇다,그렇다,매우 그렇다,그렇다,2025-03-19,3회차,True
7,2025-03-19 12:51:46,2025-03-19,3회차,박지영,디자인,1년 미만,보통이다,그렇다,그렇다,보통이다,매우 그렇다,매우 그렇다,매우 그렇다,그렇다,매우 그렇다,그렇다,2025-03-19,3회차,True
8,2025-03-26 04:00:09,2025-03-26,4회차,박민준,마케팅·광고·MD,1~3년,보통이다,그렇다,매우 그렇다,그렇지 않다,보통이다,매우 그렇다,그렇다,그렇다,그렇다,그렇다,2025-03-26,4회차,False
9,2025-03-26 08:08:03,2025-03-26,4회차,박민준,마케팅·광고·MD,1~3년,보통이다,그렇다,매우 그렇다,그렇지 않다,보통이다,매우 그렇다,그렇다,그렇다,그렇다,그렇다,2025-03-26,4회차,False


## 5. 회차 오기재 가능성 확인

- 2025-03-05 → 1회차
- 2025-03-12 → 2회차
- 2025-03-19 → 3회차
- 2025-03-26 → 4회차
- 2025-04-02 → 5회차

In [30]:
df_pre["응답시각"].astype(str).str[:10].value_counts()

응답시각
2025-03-26    25
2025-03-12    21
2025-03-05    18
2025-03-19    18
2025-04-02    16
Name: count, dtype: int64

In [31]:
df_pre[["선택회차", "응답시각"]]


,선택회차,응답시각
0,1회차,2025-03-05 01:13:19
1,1회차,2025-03-05 01:22:50
2,1회차,2025-03-05 04:42:28
3,1회차,2025-03-05 07:43:04
4,1회차,2025-03-05 08:05:20
...,...,...
93,5회차,2025-04-02 19:51:22
94,5회차,2025-04-02 19:55:23
95,5회차,2025-04-02 20:03:22
96,5회차,2025-04-02 21:29:02


### 날짜 컬럼 만들기

In [43]:
df_pre["응답일"] = pd.to_datetime(df_pre["응답시각"]).dt.date  #사전 설문
df_post["응답일"] = pd.to_datetime(df_post["응답시각"]).dt.date  #사후 설문

### 날짜 -> 회차 매핑 사전 만들기

In [35]:
DATE_TO_SESSION = {
    pd.to_datetime("2025-03-05").date(): "1회차",
    pd.to_datetime("2025-03-12").date(): "2회차",
    pd.to_datetime("2025-03-19").date(): "3회차",
    pd.to_datetime("2025-03-26").date(): "4회차",
    pd.to_datetime("2025-04-02").date(): "5회차",
}

### 응답일에 따른 정상 회차 계산

In [44]:
df_pre["정상회차"] = df_pre["응답일"].map(DATE_TO_SESSION) # 사전 설문
df_post["정상회차"] = df_post["응답일"].map(DATE_TO_SESSION) # 사후 설문

### 두 가지가 다르면 오기재로 판단

In [48]:
mismatch_pre = df_pre[df_pre["선택회차"] != df_pre["정상회차"]]

mismatch_post = df_post[df_post["선택회차"] != df_post["정상회차"]]

print(f"사전 설문 오기재 의심: {len(mismatch_pre)}")

print(f"사후 설문 오기재 의심: {len(mismatch_post)}")

사전 설문 오기재 의심: 4
사후 설문 오기재 의심: 4


### 실제 확인 후 판단하기
- 필요할 경우 회차 정상적으로 수정할 것 (전처리시)

In [50]:
display(mismatch_pre[["선택회차", "정상회차", "응답시각"]])
display(mismatch_post[["선택회차", "정상회차", "응답시각"]])

,선택회차,정상회차,응답시각
23,4회차,2회차,2025-03-12 03:59:07
60,5회차,4회차,2025-03-26 02:40:13
64,1회차,4회차,2025-03-26 03:33:04
74,2회차,4회차,2025-03-26 16:35:36


,선택회차,정상회차,응답시각
24,4회차,2회차,2025-03-12 05:29:43
61,3회차,4회차,2025-03-26 01:55:11
65,1회차,4회차,2025-03-26 05:23:02
78,3회차,4회차,2025-03-26 20:20:48


- 3월 12일: 2회차, 3월 26일: 4회차

- 응답일 기준 회차 매핑과 선택 회차를 비교한 결과
    - 사전 설문: 4건
    - 사후 설문: 4건
- 총 8건의 회차 오기재 의심 응답이 확인됨.

## 6. 직무 문자열 정규화

In [ ]:
df_pre["직무"].unique() # 사전

array(['마케팅·광고·MD', '디자인', '마케팅/광고', '기획·전략', '마케팅', '전략기획', '기타/디자인',
       '영업·영업지원', '기획 전략', '기타 - 마케팅', '마케팅 광고', '기타(영업)', '기타 : 마케팅·광고',
       '기타(기획·전략)', '디자인 ', '기획전략'], dtype=object)

In [ ]:
df_post["직무"].unique() # 사후

array(['기획·전략', '디자인', '마케팅광고', '마케팅·광고·MD', '기타 : 마케팅·광고', '영업·영업지원',
       '기획전략', '마케팅 광고', '기타 - 마케팅', '기획 전략', 'MD', '기획/전략', '마케팅',
       '광고·MD', '기타/디자인', '디자인 ', '기타', '기타(기획·전략)'], dtype=object)

In [68]:
def normalize_job(x):
    x = str(x).strip()
    
    if "마케팅" in x or "광고" in x or "MD" in x:
        return "마케팅"
    elif "기획" in x or "전략" in x:
        return "기획/전략"
    elif "디자인" in x:
        return "디자인"
    elif "영업" in x:
        return "영업"
    else:
        return "기타"

df_pre["직무_정규"] = df_pre["직무"].apply(normalize_job)
df_post["직무_정규"] = df_post["직무"].apply(normalize_job)

In [62]:
df_pre["직무_정규"].value_counts()

직무_정규
마케팅      37
기획/전략    28
디자인      20
영업       13
Name: count, dtype: int64

In [69]:
df_post["직무_정규"].value_counts()

직무_정규
마케팅      38
기획/전략    28
디자인      20
영업       11
기타        1
Name: count, dtype: int64

In [72]:
df_post[df_post["직무_정규"] == '기타']

,응답시각,교육일,선택회차,참여자명,직무,엑셀사용경력,A1_데이터정리수치도출,A2_데이터시각화,B1_업무활용계획,B2_교육내용적용의향,C1_팀내공유의향,C2_데이터문화정착가능성,D1_강사,D2_내용,D3_운영환경,D4_전반만족도,응답일,정상회차,직무_정규
75,2025-03-26 18:24:12,2025-03-26,4회차,류나은,기타,3년 이상,보통이다,보통이다,매우 그렇다,매우 그렇다,그렇다,보통이다,보통이다,보통이다,그렇다,그렇다,2025-03-26,4회차,기타


### 동명이인 여부 다시 확인

In [76]:
dup_pre = df_pre[df_pre.duplicated(subset=["선택회차", "참여자명"], keep=False)]

job_check = (
    dup_pre.groupby(["선택회차", "참여자명"])["직무_정규"]
       .nunique()
       .reset_index(name="job_unique")
)

job_check["is_different_person"] = job_check["job_unique"] > 1

In [77]:
dup_pre = dup_pre.merge(
    job_check[["선택회차", "참여자명", "is_different_person"]],
    on=["선택회차", "참여자명"],
    how="left"
)
dup_pre

,응답시각,교육일,선택회차,참여자명,직무,엑셀사용경력,A1_데이터정리수치도출,A2_데이터시각화,B1_업무활용계획,B2_교육내용적용의향,C1_팀내공유의향,C2_데이터문화정착가능성,응답일,정상회차,직무_정규,is_different_person
0,2025-03-12 00:48:44,2025-03-12,2회차,황예준,디자인,3년 이상,보통이다,그렇지 않다,매우 그렇다,매우 그렇다,보통이다,그렇지 않다,2025-03-12,2회차,디자인,True
1,2025-03-12 01:04:47,2025-03-12,2회차,이준서,기획·전략,1년 미만,보통이다,전혀 그렇지 않다,그렇지 않다,전혀 그렇지 않다,전혀 그렇지 않다,그렇다,2025-03-12,2회차,기획/전략,False
2,2025-03-12 06:26:04,2025-03-12,2회차,황예준,마케팅·광고·MD,거의 없음,전혀 그렇지 않다,그렇지 않다,그렇지 않다,그렇다,그렇지 않다,보통이다,2025-03-12,2회차,마케팅,True
3,2025-03-12 09:47:56,2025-03-12,2회차,이준서,기획·전략,1년 미만,보통이다,전혀 그렇지 않다,그렇지 않다,전혀 그렇지 않다,전혀 그렇지 않다,그렇다,2025-03-12,2회차,기획/전략,False
4,2025-03-19 01:47:06,2025-03-19,3회차,홍민재,마케팅 광고,거의 없음,그렇지 않다,그렇지 않다,그렇지 않다,보통이다,보통이다,그렇다,2025-03-19,3회차,마케팅,False
5,2025-03-19 04:48:01,2025-03-19,3회차,박지영,영업·영업지원,3년 이상,보통이다,보통이다,매우 그렇다,매우 그렇다,보통이다,매우 그렇다,2025-03-19,3회차,영업,True
6,2025-03-19 15:32:03,2025-03-19,3회차,홍민재,마케팅,거의 없음,보통이다,보통이다,그렇지 않다,전혀 그렇지 않다,전혀 그렇지 않다,그렇지 않다,2025-03-19,3회차,마케팅,False
7,2025-03-19 21:14:55,2025-03-19,3회차,박지영,디자인,1년 미만,보통이다,보통이다,보통이다,보통이다,보통이다,보통이다,2025-03-19,3회차,디자인,True
8,2025-03-26 01:10:46,2025-03-26,4회차,박민준,마케팅·광고·MD,1~3년,그렇지 않다,보통이다,그렇다,전혀 그렇지 않다,그렇지 않다,그렇다,2025-03-26,4회차,마케팅,False
9,2025-03-26 02:40:34,2025-03-26,4회차,박민준,마케팅·광고·MD,1~3년,그렇지 않다,보통이다,그렇다,전혀 그렇지 않다,그렇지 않다,그렇다,2025-03-26,4회차,마케팅,False


- '홍민재-마케팅', '홍민재-마케팅광고'가 같은 사람(is different person = false)로 나옴을 확인 

In [78]:
dup_post = df_post[df_post.duplicated(subset=["선택회차", "참여자명"], keep=False)]

job_check = (
    dup_post.groupby(["선택회차", "참여자명"])["직무_정규"]
       .nunique()
       .reset_index(name="job_unique")
)

job_check["is_different_person"] = job_check["job_unique"] > 1

In [79]:
dup_post = dup_post.merge(
    job_check[["선택회차", "참여자명", "is_different_person"]],
    on=["선택회차", "참여자명"],
    how="left"
)
dup_post

,응답시각,교육일,선택회차,참여자명,직무,엑셀사용경력,A1_데이터정리수치도출,A2_데이터시각화,B1_업무활용계획,B2_교육내용적용의향,C1_팀내공유의향,C2_데이터문화정착가능성,D1_강사,D2_내용,D3_운영환경,D4_전반만족도,응답일,정상회차,직무_정규,is_different_person
0,2025-03-12 03:59:20,2025-03-12,2회차,이준서,기획·전략,1년 미만,매우 그렇다,그렇지 않다,그렇지 않다,전혀 그렇지 않다,그렇지 않다,매우 그렇다,그렇다,그렇다,그렇다,보통이다,2025-03-12,2회차,기획/전략,False
1,2025-03-12 12:55:52,2025-03-12,2회차,황예준,마케팅·광고·MD,거의 없음,그렇지 않다,보통이다,보통이다,그렇다,보통이다,매우 그렇다,그렇다,보통이다,그렇다,그렇다,2025-03-12,2회차,마케팅,True
2,2025-03-12 15:16:20,2025-03-12,2회차,황예준,디자인,3년 이상,그렇다,보통이다,매우 그렇다,그렇다,그렇다,보통이다,보통이다,매우 그렇다,매우 그렇다,그렇다,2025-03-12,2회차,디자인,True
3,2025-03-12 16:08:14,2025-03-12,2회차,이준서,기획·전략,1년 미만,매우 그렇다,그렇지 않다,그렇지 않다,전혀 그렇지 않다,그렇지 않다,매우 그렇다,그렇다,그렇다,그렇다,보통이다,2025-03-12,2회차,기획/전략,False
4,2025-03-19 00:37:11,2025-03-19,3회차,홍민재,마케팅,거의 없음,그렇다,보통이다,보통이다,그렇다,그렇다,그렇다,매우 그렇다,그렇다,그렇다,그렇다,2025-03-19,3회차,마케팅,False
5,2025-03-19 02:16:41,2025-03-19,3회차,홍민재,광고·MD,거의 없음,그렇다,매우 그렇다,그렇다,그렇지 않다,전혀 그렇지 않다,보통이다,그렇다,그렇다,그렇다,그렇다,2025-03-19,3회차,마케팅,False
6,2025-03-19 07:18:28,2025-03-19,3회차,박지영,영업·영업지원,3년 이상,그렇다,매우 그렇다,매우 그렇다,매우 그렇다,그렇다,매우 그렇다,매우 그렇다,그렇다,매우 그렇다,그렇다,2025-03-19,3회차,영업,True
7,2025-03-19 12:51:46,2025-03-19,3회차,박지영,디자인,1년 미만,보통이다,그렇다,그렇다,보통이다,매우 그렇다,매우 그렇다,매우 그렇다,그렇다,매우 그렇다,그렇다,2025-03-19,3회차,디자인,True
8,2025-03-26 04:00:09,2025-03-26,4회차,박민준,마케팅·광고·MD,1~3년,보통이다,그렇다,매우 그렇다,그렇지 않다,보통이다,매우 그렇다,그렇다,그렇다,그렇다,그렇다,2025-03-26,4회차,마케팅,False
9,2025-03-26 08:08:03,2025-03-26,4회차,박민준,마케팅·광고·MD,1~3년,보통이다,그렇다,매우 그렇다,그렇지 않다,보통이다,매우 그렇다,그렇다,그렇다,그렇다,그렇다,2025-03-26,4회차,마케팅,False


## 7. 현재까지 작업한 내용 파일로 저장

In [80]:
OUTPUT_PATH = "../data/stage1_clean.xlsx"

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df_pre.to_excel(writer, sheet_name="사전_정제", index=False)
    df_post.to_excel(writer, sheet_name="사후_정제", index=False)

print("저장 완료")

저장 완료


## 8. 전처리 정책 정의

### 전처리 원칙
- 원본 데이터는 수정하지 않는다.
- 제거 및 교정된 데이터 건수는 기록한다.
- 동일 데이터가 추가될 경우 동일 로직을 반복 적용할 수 있도록 설계한다.

### 전처리 세부 정책

| 구분 | 대상 컬럼 | 처리 규칙 | 목적 |
|------|------------|------------|------|
| 데이터 타입 정규화 | 응답시각 | datetime 형식으로 변환, 응답일 컬럼 생성 | 회차 검증 및 시간 기준 정렬 |
| 중복 응답 처리 | 선택회차 + 참여자명 | 동일 조합 발생 시 최신 응답만 유지 | 중복 데이터 제거 |
| 동일 인물 판별 | is_different_person | False인 경우 동일 인물로 간주 | 잘못된 중복 분리 방지 |
| 회차 교정 | 선택회차 | 응답일 기준 정상회차 계산 후 교정, `회차` 컬럼 통합 생성 | 오기재 수정 |
| 리커트 변환 | A, B, C 영역 문항 | 텍스트 → 1~5 숫자형 변환, 매핑 실패 시 NaN | 수치 분석 가능화 |
| 익명화 | 참여자명 | 참여자명 제거 후 고유 ID 생성 | 개인정보 보호 |
| 사전·사후 매칭 | 회차 + 참여자명 | 둘 중 하나만 존재 시 분석 제외 | 대응표본 분석 가능화 |
| 결측 처리 | 핵심 문항 | 핵심 문항 결측 시 분석 제외 | 통계 검정 정확성 확보 |
| 파생 지표 생성 | 영역 점수 | 사전/사후 평균 점수 생성 | 분석 편의성 향상 |